In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Phase 6 — Practical Model Implementation & Behavior Testing

## Overview

Hereby, we are going to test the estimated models on realistic policy scenarios to check their practical implementation. Discrete choice models are only useful if they produce sensible predictions when inputs change — a model that fits historical data well but produces implausible forecasts under interventions is not fit for policy analysis.

We implement two policy scenarios drawn from current urban transport debates in Europe:

1. London — Cycle Infrastructure Expansion: A 20% reduction in the effective distance penalty for cycling, representing investment in protected bike lanes, secure parking, and bike-sharing stations. London currently has low cycling mode share (8%).

2. Amsterdam — Congestion Charge: A hypothetical €5/day charge on car commuting into central Amsterdam (modelled as an implicit income reduction for car users), similar to schemes in Stockholm and Milan. This tests whether Amsterdam's car users would shift to cycling or PT under economic disincentives.

For each scenario we calculate:

- Aggregate mode shift — change in predicted mode shares before/after the intervention
- Equity impact — distributional effects across income quintiles (who benefits?)

In [17]:
amst_data = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\amst_processed.csv")

In [3]:
amst_final_beta = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\amst_final_beta.csv")

In [4]:
lnd_data = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\lnd_processed.csv")

In [5]:
lnd_final_beta = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\lnd_final_beta.csv")

## 4.9.1 Mode Choice Prediction Function

The core prediction task is: given an individual's characteristics and a trip's level-of-service attributes, compute the probability they choose each mode. The MNL choice probability for alternative $j$ is:

$$P(j \mid \mathbf{x}) = \frac{\exp(V_j)}{\sum_{k \in \{car, PT, bike\}} \exp(V_k)}$$

where $V_j = \beta_j' \mathbf{x}$ is the systematic utility. We extract the estimated coefficients from `amst_final_beta` and `lnd_final_beta`, construct the three utility expressions, apply the softmax transformation, and return a dictionary of probabilities.

In [32]:
def get_coef(city_beta, name):
    row = city_beta.loc[city_beta["Name"] == name]
    return row["Value"].values[0] if not row.empty else 0.0

def predict_mode_choice(city_beta, travel_time_min, distance_km,
                        income_quintile, age_band, gender, is_peak=0):
    g = lambda name: get_coef(city_beta, name)

    v_pt = (g("asc_pt")
            + g("b_time_pt")    * travel_time_min
            + g("b_income_pt")  * income_quintile
            + g("b_age_pt")     * age_band
            + g("b_gender_pt")  * gender
            + g("b_peak_pt")    * is_peak)

    v_bike = (g("asc_bike")
              + g("b_time_bike")   * travel_time_min
              + g("b_income_bike") * income_quintile
              + g("b_age_bike")    * age_band
              + g("b_gender_bike") * gender
              + g("b_dist_bike")   * distance_km
              + g("b_peak_bike")   * is_peak)

    exp_v = np.exp([0.0, v_pt, v_bike])
    probs = exp_v / exp_v.sum()
    return dict(zip(("car", "pt", "bike"), probs))

def predict_row(row, city_beta, dist_multiplier=1.0):
    return predict_mode_choice(
        city_beta,
        travel_time_min  = row["travel_time_min"],
        distance_km      = row["distance_km"] * dist_multiplier,
        income_quintile  = row["income_quintile"],
        age_band         = row["age_band"],
        gender           = row["gender"],
        is_peak          = row["is_peak"],
    )

def validate_model(city_beta, city_workset, n_sample=100):
    sample = city_workset.sample(n=min(n_sample, len(city_workset)), random_state=42)

    records = []
    for _, row in sample.iterrows():
        probs = predict_row(row, city_beta)
        predicted = max(probs, key=probs.get)
        actual = row["chosen_mode"]
        records.append({
            "actual":    actual,
            "predicted": predicted,
            **{f"p_{m}": probs[m] for m in ("car", "pt", "bike")},
            "correct":   predicted == actual,
        })

    results = pd.DataFrame(records)
    hit_rate = results["correct"].mean()
    mean_prob_chosen = results.apply(lambda r: r[f"p_{r['actual']}"], axis=1).mean()
    return {"hit_rate": hit_rate, "mean_prob_chosen": mean_prob_chosen, "predictions": results}

def print_validation(city_name, val):
    preds  = val["predictions"]
    cm     = pd.crosstab(preds["predicted"], preds["actual"])
    modes  = cm.columns.tolist()
    col_w  = 10

    print(f"\n{city_name}")
    print(f"  Sample:{len(preds):,}")
    print(f"  Hit rate: {val['hit_rate']:.1%}")
    print(f"  Mean P(chosen){val['mean_prob_chosen']:>7.3f}")

    # Actual vs predicted per mode
    n = len(preds)
    print(f"\n  {'Mode':<10}{'Actual':>10}{'Predicted':>16}{'Diff':>10}")
    for m in modes:
        actual_n    = (preds["actual"]    == m).sum()
        predicted_n = (preds["predicted"] == m).sum()
        diff        = predicted_n - actual_n
        print(f"  {m:<10}{actual_n:>6} ({actual_n/n:.0%})"
              f"  {predicted_n:>6} ({predicted_n/n:.0%})"
              f"  {diff:>+7}")

In [33]:
# Validation
print_validation("AMSTERDAM", validate_model(amst_final_beta, amst_data,  n_sample=100))
print_validation("LONDON", validate_model(lnd_final_beta,  lnd_data,   n_sample=200))


AMSTERDAM
  Sample:100
  Hit rate: 80.0%
  Mean P(chosen)  0.669

  Mode          Actual       Predicted      Diff
  bike          63 (63%)      69 (69%)       +6
  car           28 (28%)      25 (25%)       -3
  pt             9 (9%)       6 (6%)       -3

LONDON
  Sample:200
  Hit rate: 63.0%
  Mean P(chosen)  0.521

  Mode          Actual       Predicted      Diff
  bike          18 (9%)       0 (0%)      -18
  car           76 (38%)      86 (43%)      +10
  pt           106 (53%)     114 (57%)       +8


From the above validaiton step we can see the accuracy of our model on a radnom sample set derived from the actual Amsterdam and London data. 

1. Amsterdam Model Performance - the hit rate of the model is 80% meaning that out of 100 actual trips, the model correctly predicted the chosen mode 80 times.On average, the model was 66.9% confident about its prediction for the mode people actually chose. Regarding the mode choice, as expected the model captures bike choice well but underestimates the public transport and cars.
2. London Model Performance - the hit rate of the Londons model is lower - 63% as out of 200 actual trips, the model correctly predicted the chosen mode 126 times. On average, the model was only 52.1% confident (barely better than a coin flip). In Londons case the model strongly ignores the bike as a choice and overestimate the cars and pt usage. This is expected as London has more diverse transport options and more complex decision patterns than Amsterdam's bike-dominated system. Yet, completely ignoring bike mode choice is a strong model limitation which needs to be handled. 

## Scenario 1 — London Cycle Infrastructure Expansion

Policy: Invest £500M in protected cycle lanes, secure bike parking at rail stations, and expansion of the Santander Cycles bike-share scheme. We model this as a 20% reduction in the effective distance penalty for cycling — equivalent to making a 10km cycle trip "feel like" 8km due to improved infrastructure quality and safety.

Implementation: Multiply `b_dist_bike` by 0.8 in the utility function for bicycle. We then re-predict mode shares for a representative sample of 1,000 London commuters drawn from the `lnd_data`, calculate the shift in aggregate mode shares, and decompose by income quintile to assess equity.

> Hypothesis: Bike infrastructure investment will increase cycling mode share, with larger effects among higher-income groups who already own bicycles and can capitalise on improved infrastructure.

In [12]:
def weighted_mean(series, weights):
    return (series * weights).sum() / weights.sum()

def mode_shares(df, period):
    w = df["weight_trip"]
    return {m: weighted_mean(df[f"p_{m}_{period}"], w) for m in ("car", "pt", "bike")}
    
def apply_scenario(sample, city_beta, after_fn):
    sample = sample.copy()
    sample["pred_before"] = sample.apply(lambda r: predict_row(r, city_beta), axis=1)
    sample["pred_after"]  = sample.apply(lambda r: after_fn(r, city_beta),    axis=1)
    for period in ("before", "after"):
        for mode in ("car", "pt", "bike"):
            sample[f"p_{mode}_{period}"] = sample[f"pred_{period}"].apply(lambda x: x[mode])
    return sample

In [13]:
def print_scenario(title, sample):
    before = mode_shares(sample, "before")
    after  = mode_shares(sample, "after")
    shifts = {m: after[m] - before[m] for m in ("car", "pt", "bike")}

    print(f"\n{title}")
    print(f"  Aggregate Mode Shares  (n={len(sample):,})")
    print(f"  {'Mode':<8}{'Before':>8}{'After':>8}{'Shift':>8}")
    for m in ("car", "pt", "bike"):
        print(f"  {m.capitalize():<8}{before[m]:>8.1%}{after[m]:>8.1%}{shifts[m]:>+8.1%}")

    print(f"\n  Equity Impact by Income Quintile")
    print(f"  {'Quintile':<14}{'Car':>10}{'PT':>10}{'Bike':>10}")
    for q in range(1, 6):
        qs = sample[sample["income_quintile"] == q]
        if qs.empty:
            continue
        qb = mode_shares(qs, "before")
        qa = mode_shares(qs, "after")
        qs = {m: qa[m] - qb[m] for m in ("car", "pt", "bike")}
        print(f"  Q{q} (n={len(sample[sample['income_quintile']==q]):3d})     "
              f"{qs['car']:>+8.2%}  {qs['pt']:>+8.2%}  {qs['bike']:>+8.2%}")

In [14]:
# Scenario 1 — London: Cycle Infrastructure (20% distance reduction)
np.random.seed(42)
lnd_sample = lnd_data.sample(n=min(1000, len(lnd_data)), random_state=42)
print_scenario(
    "SCENARIO 1  London Cycle Infrastructure  (20% distance reduction)",
    apply_scenario(lnd_sample, lnd_final_beta,
                   after_fn=lambda r, b: predict_row(r, b, dist_multiplier=0.8))
)


SCENARIO 1  London Cycle Infrastructure  (20% distance reduction)
  Aggregate Mode Shares  (n=1,000)
  Mode      Before   After   Shift
  Car        36.8%   36.1%   -0.8%
  Pt         54.3%   53.3%   -1.0%
  Bike        8.9%   10.6%   +1.7%

  Equity Impact by Income Quintile
  Quintile             Car        PT      Bike
  Q1 (n=149)       -0.44%    -0.49%    +0.94%
  Q2 (n=133)       -0.44%    -0.58%    +1.03%
  Q3 (n=142)       -0.77%    -0.76%    +1.52%
  Q4 (n=217)       -0.85%    -0.99%    +1.83%
  Q5 (n=359)       -0.98%    -1.51%    +2.49%


In this scenario we tested a scenario on how a 20% reduction in cycle distance (new bike lanes, shorter routes) would affect mode choice among diffent income groups. 

Generally, Bike mode choice grows by 1.7 percentage points (from 8.9% → 10.6%) and PT loses 1.0 pp — people switch from PT to bike.
Car usage also drops by 0.8 pp — some car users switch to bike. Eventually, we can observe a modest but positive shift toward active commuting - biking. We can suggest that better biking infrastructure makes this choice more attractive. People who switch are mainly from PT (+1.0%) rather than cars (+0.8%). We assume that the shift is relatively small because most Londoners face other barriers to biking to work beyond just route distance.

Regarding the mode shift among different income groups, interestingly this policy would benefit more rich people than the low income group. Q1 actually experience the lowest mode shift while Q5 (the highest income group) has the biggest shift. Why would this be the case? We can suggest that wealthier commuters usually live in safer neighborhoods with existing bike infrastructure and can afford better bikes. On the contrary, poorer commuters might not have access to safe cycling infrastructure and less flexible commuting options. To support these groups of commuters we can combine the tested policy with subsidized bike-share schemes for Q1-or community cycling programs in underserved area.   

## 4.9.3 Scenario 2 — Amsterdam Congestion Charge

Policy: Introduce a €5/day charge for cars entering the central city, operational Mon-Fri 7am - 7pm. We model this as an implicit income reduction for car users — a traveller with monthly income of €3,000 who drives 20 days/month pays €100, equivalent to a 3.3% income reduction. Since our model does not include an explicit cost variable, we approximate the effect by reducing the income quintile by 0.5 units for car users.

Implementation: For each sampled commuter, if their baseline prediction assigns >50% probability to car, we reduce their `income_quintile` by 0.5 and re-predict. This simulates the deterrent effect of the charge on car use.

> Hypothesis: The charge will shift car users to cycling (Amsterdam's dominant alternative) more than to PT, with weaker effects among higher-income groups who can absorb the cost.

In [15]:
# Scenario 2 — Amsterdam: Congestion Charge (€5/day)
def congestion_charge(row, city_beta):
    baseline = predict_row(row, city_beta)
    if baseline["car"] > 0.5:
        return predict_mode_choice(
            city_beta,
            travel_time_min = row["travel_time_min"],
            distance_km     = row["distance_km"],
            income_quintile = max(1, row["income_quintile"] - 0.5),
            age_band        = row["age_band"],
            gender          = row["gender"],
            is_peak         = row["is_peak"],
        )
    return baseline

In [16]:
ams_sample = amst_data.sample(n=min(500, len(amst_data)), random_state=42)
print_scenario(
    "SCENARIO 2  Amsterdam Congestion Charge  (€5/day)",
    apply_scenario(ams_sample, amst_final_beta, after_fn=congestion_charge)
)


SCENARIO 2  Amsterdam Congestion Charge  (€5/day)
  Aggregate Mode Shares  (n=500)
  Mode      Before   After   Shift
  Car        25.1%   24.9%   -0.2%
  Pt         13.5%   13.8%   +0.3%
  Bike       61.4%   61.3%   -0.1%

  Equity Impact by Income Quintile
  Quintile             Car        PT      Bike
  Q1 (n= 62)       +0.00%    +0.00%    +0.00%
  Q2 (n= 63)       -0.12%    +0.16%    -0.04%
  Q3 (n= 80)       -0.30%    +0.40%    -0.10%
  Q4 (n=130)       -0.20%    +0.34%    -0.14%
  Q5 (n=165)       -0.35%    +0.46%    -0.11%


In this scenario we test how the introduction of congestion penalty on Amsterdam car commuters (5 euro daily charge). Interestingly, we see that car choice barely drops (only -0.2%). PT gains tiny +0.3% and bike choice remains unchanged (-0.1%). We can conclude that this policy has minimal to no effect on commuters mode choice. Amsterdam is already 61% bike dominant. People who don't drive already don't drive. The remaining 25% car users must be essential drivers or have high willingness-to-pay who view €5 as acceptable.

Regarding the change on the different income commuters, we see a slight regressive effect. The charge only affects people rich enough to drive. Amsterdammers with lower income are already on bikes/PT and unaffected. A congestion charge of 5 euro is not efficient enough to redistribute the mode share towards biking. If want to support mode shift towards PT or biking Amsterdam municipality can try Other measures as parking restrictions, delivery windows or focus on free/cheap PT for low-income users instead.